In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "00-foundations/gpu-capacity-planning/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# LLM GPU Capacity — Practice

**Solutions.**

## Setup (given — don't change)
The only constants you need. Everything else you'll derive.

In [ ]:
BYTES = {"bf16": 2.0, "fp8": 1.0, "int4": 0.5}   # bytes per parameter / KV element

# GPU: HBM GB, bandwidth TB/s, bf16 TFLOPS, fp8 TFLOPS
GPU = {
    "H100": dict(hbm=80,  bw=3.35, bf16=990, fp8=1979),
    "H200": dict(hbm=141, bw=4.80, bf16=990, fp8=1979),
}

# Read these from a real model's config.json. Mistral Small 3 (24B dense):
SMALL = dict(params_b=24, layers=40, kv_heads=8, head_dim=128, active_b=24)
# Mistral Large 3 (675B MoE, 41B active):
LARGE = dict(params_b=675, layers=88, kv_heads=8, head_dim=128, active_b=41)
print("ready")

## Reference for the checks (given — don't change)
The checks below compare your functions with `capacity.py`, the primer's reference code, on several inputs,
so a constant or a wrong formula fails. Memory is in GB = 10⁹ bytes throughout (KV per token in kB = 10³ bytes),
as in `capacity.py`.

In [ ]:
import math, pathlib, sys
_dirs = [pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd() / "00-foundations" / "gpu-capacity-planning"]
sys.path.insert(0, str(next(d for d in _dirs if (d / "capacity.py").exists())))
import capacity as ref                         # the primer's reference code (../capacity.py)

def ref_model(m):                              # SMALL / LARGE dicts -> capacity.ModelSpec
    return ref.ModelSpec("model", **m)

def close(got, want):                          # same number to 6 significant digits
    return math.isclose(got, want, rel_tol=1e-6)
print("reference loaded:", ref.__file__)

## 1. Weights — does it fit?
`memory = params × bytes/param`.

In [ ]:
def weight_gb(params_b, dtype):
    return params_b * BYTES[dtype]

In [ ]:
assert weight_gb(24, "bf16") == 48
assert weight_gb(24, "fp8")  == 24
assert weight_gb(24, "int4") == 12
print("weights:", weight_gb(24,"bf16"), "GB bf16 |", weight_gb(24,"fp8"), "GB fp8")

## 2. KV cache per token — the concurrency tax
`KV/token = 2 × layers × kv_heads × head_dim × bytes`. Return **kB** (1 kB = 1,000 bytes).
It uses `kv_heads` (8), not the 32 query heads — that's GQA saving 4×.

In [ ]:
def kv_per_token_kb(m, dtype):
    return 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * BYTES[dtype] / 1000

In [ ]:
for m in (SMALL, LARGE):
    for dt in ("bf16", "fp8", "int4"):
        assert close(kv_per_token_kb(m, dt), ref.kv_per_token_kb(ref_model(m), dt)), (m, dt)
assert close(kv_per_token_kb(SMALL, "bf16"), 163.84)   # 163,840 bytes = 163.84 kB (1 kB = 1,000 bytes)
print("KV/token:", kv_per_token_kb(SMALL,"bf16"), "kB (bf16)")

## 3. How many concurrent users fit?
Spare HBM after weights, divided by KV per conversation. `context_tokens` long.
KV per conversation in GB = kB/token × tokens / 10⁶ — the same GB as the weights and the HBM.

In [ ]:
def max_sessions(spare_gb, m, context_tokens, dtype):
    kv_session_gb = kv_per_token_kb(m, dtype) * context_tokens / 1e6
    return spare_gb / kv_session_gb

In [ ]:
for spare in (24, 48):
    for ctx in (1650, 8000, 32000):
        for dt in ("bf16", "fp8"):
            want = ref.max_concurrent_sessions(spare, ref_model(SMALL), ctx, dt)
            assert close(max_sessions(spare, SMALL, ctx, dt), want), (spare, ctx, dt)
usable = 80 * 0.9                              # H100 minus ~10% overhead
n_bf16 = max_sessions(usable - weight_gb(24,"bf16"), SMALL, 8000, "bf16")
n_fp8  = max_sessions(usable - weight_gb(24,"fp8"),  SMALL, 8000, "fp8")
print(f"8K sessions/H100:  bf16 ~{n_bf16:.0f}   fp8 ~{n_fp8:.0f}  (fp8 = 4x lever)")

## 4. Decode speed — bandwidth-bound
Each token streams all weights through HBM. `tok/s = 1/(weight_GB/bw)`.
Units: 1 TB/s = 1000 GB/s.

In [ ]:
def decode_tok_s(weight_gb_, gpu):
    step_s = weight_gb_ / (gpu["bw"] * 1000)
    return 1 / step_s

In [ ]:
for w_gb in (12, 24, 48, 140):
    for name in ("H100", "H200"):
        want = ref.decode_tok_s_single(w_gb, ref.GPUS[name])
        assert close(decode_tok_s(w_gb, GPU[name]), want), (w_gb, name)
t = decode_tok_s(weight_gb(24,"bf16"), GPU["H100"])
print(f"single-stream ceiling ~{t:.0f} tok/s  (more COMPUTE won't raise this)")

## 5. Prefill — compute-bound → TTFT
`FLOPs = 2 × params × prompt_tokens`, `TTFT = FLOPs/(peak×MFU)`.
Use **active** params (matters for MoE), fp8 peak, MFU 0.5.

In [ ]:
def ttft_s(active_b, prompt_tokens, gpu, mfu=0.5):
    flops = 2 * active_b * 1e9 * prompt_tokens
    return flops / (gpu["fp8"] * 1e12 * mfu)

In [ ]:
for active_b in (24, 41):
    for prompt in (500, 2000, 32000):
        for name in ("H100", "H200"):
            for mfu in (0.5, 0.4):
                want = ref.ttft_s(active_b, prompt, ref.GPUS[name], "fp8", mfu)   # weights only, fp8 peak
                assert close(ttft_s(active_b, prompt, GPU[name], mfu), want), (active_b, prompt, name, mfu)
print(f"TTFT 2K ~{ttft_s(24,2000,GPU['H100']):.2f}s   32K ~{ttft_s(24,32000,GPU['H100']):.2f}s")

## 6. Workload → concurrency (Little's Law)
`duration = TTFT + output × TPOT`, `concurrency = RPS × duration`.

In [ ]:
def concurrency(rps, active_b, in_tok, out_tok, gpu, tpot_ms=40):
    duration = ttft_s(active_b, in_tok, gpu) + out_tok * tpot_ms / 1000
    return rps * duration

In [ ]:
for rps in (1.0, 8.3):
    for active_b, in_tok, out_tok, tpot_ms in ((24, 1500, 300, 40), (24, 8000, 3000, 20), (41, 2000, 500, 50)):
        want = rps * ref.request_duration_s(active_b, in_tok, out_tok, ref.GPUS["H100"], tpot_ms)
        assert close(concurrency(rps, active_b, in_tok, out_tok, GPU["H100"], tpot_ms), want), (rps, active_b, in_tok)
c = concurrency(8.3, 24, 1500, 300, GPU["H100"])
print(f"~{c:.0f} concurrent sessions from 8.3 RPS")

## 7. Capstone — size the bank
Uses the functions you wrote above. Compute GPUs by each constraint; take the max.
Nothing to fill here — just run it once 1–6 pass.

In [ ]:
import math
conc = concurrency(8.3, 24, 1500, 300, GPU["H100"])
avg_ctx = 1500 + 300 // 2
usable = 80 * 0.9

g_mem_fp8  = conc / max_sessions(usable - weight_gb(24,"fp8"),  SMALL, avg_ctx, "fp8")
g_mem_bf16 = conc / max_sessions(usable - weight_gb(24,"bf16"), SMALL, avg_ctx, "bf16")
prefill_cap = 0.5 * GPU["H100"]["fp8"] * 1e12 / (2 * 24e9)   # prefill tok/s per GPU
prefill_gpus = (8.3 * 1500) / prefill_cap
raw = max(g_mem_fp8, prefill_gpus)

per_gpu_fp8  = max_sessions(usable - weight_gb(24,"fp8"),  SMALL, avg_ctx, "fp8")
per_gpu_bf16 = max_sessions(usable - weight_gb(24,"bf16"), SMALL, avg_ctx, "bf16")
print(f"sessions/GPU fp8: {per_gpu_fp8:.1f}   bf16: {per_gpu_bf16:.1f}   (at {avg_ctx:,} tokens of context)")
print(f"memory   fp8: {g_mem_fp8:.1f} GPU   bf16: {g_mem_bf16:.1f} GPU  <- the driver")
print(f"prefill tput: {prefill_gpus:.1f} GPU")
print(f"raw (fp8) {raw:.1f} -> provision 2 (1 active + 1 spare); buy an 8-GPU node")
assert g_mem_bf16 > g_mem_fp8
assert (round(per_gpu_fp8, 1), round(per_gpu_bf16, 1)) == (355.1, 88.8)   # capacity.py's bank example

## 8. Stretch — why Mistral Large 3 breaks a single node
MoE: **memory** by TOTAL params, **compute** by ACTIVE params.

In [ ]:
w = weight_gb(675, "fp8")                     # 675 GB
fits_8xh100 = w < 8 * GPU["H100"]["hbm"] * 0.9
fits_8xh200 = w < 8 * GPU["H200"]["hbm"] * 0.9

assert w == 675
assert fits_8xh100 is False                   # 576 GB usable < 675
assert fits_8xh200 is True                    # ~1 TB usable
print(f"Large 3 fp8 = {w} GB | fits 8xH100? {fits_8xh100} | 8xH200? {fits_8xh200}")
print("prefill sized by 41B active, not 675B total")